In [25]:
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages

from langgraph.prebuilt import ToolNode, tools_condition

from langchain.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

from langchain_ollama import ChatOllama

from typing import TypedDict, Annotated

from dotenv import load_dotenv
import os
from tavily import TavilyClient


In [26]:
%pip install tavily-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\chimu\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [27]:
load_dotenv()
ollama_api_key = os.getenv("OLLAMA_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

In [28]:
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage], add_messages]

In [29]:
llm=ChatOllama(
    model="gemma4",
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": f"Bearer {ollama_api_key}"}},
)

In [30]:
@tool
def search_tool(query: str) -> dict:
    """Searches the web for the given query."""
    client = TavilyClient(tavily_api_key)
    response = client.search(query=query,search_depth="advanced")
    return response

In [31]:
@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}
        
        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    except Exception as e:
        return {"error": str(e)}

In [40]:
# Make tool list
tools=[search_tool,calculator]

In [33]:
# Make the LLM tool-aware
llm_with_tools=llm.bind_tools(tools)

In [48]:
#define constants
CHATNODE="chat_node"
TOOLS="tools"

In [49]:
# graph nodes
def chat_node(state:ChatState)->ChatState:
    """ LLM node that may answer a question or request a tool call"""
    messages=state['messages']
    response=llm_with_tools.invoke(messages)
    state['messages'].append(response)
    return state

In [50]:
tool_node=ToolNode(tools) # Executes tool calls

In [51]:
# graph structure
graph=StateGraph(ChatState)
# add nodes to graph
graph.add_node(CHATNODE,chat_node)
graph.add_node(TOOLS,tool_node)

#add edges to graph
graph.add_edge(START,CHATNODE)

# If the LLM asked for a tool, go to ToolNode; else finish
graph.add_conditional_edges(CHATNODE,tools_condition)

graph.add_edge(CHATNODE,TOOLS)   # loop through graph

In [52]:
chatbot=graph.compile()

In [53]:
# Regular chat
out = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]})

print(out["messages"][-1].content)

Hello! How can I help you today?


In [54]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is 2*3?")]})
print(out["messages"][-1].content)

{"first_num": 2.0, "second_num": 3.0, "operation": "mul", "result": 6.0}


In [55]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is the stock price of apple")]})
print(out["messages"][-1].content)

{"query": "Apple stock price", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.investing.com/equities/apple-computer-inc-historical-data", "title": "Apple Stock Price History - Investing.com", "content": "Economic CalendarStock Screener\n   \n\nNASDAQ\n\n|  |\n\n|  | Symbol |  | Exchange |  | Currency |\n|  | AAPL | · | NASDAQ | · | USD | Real-time |\n|  | 0R2V | · | London | · | USD | Real-time |\n|  | AAPL | · | Xetra | · | EUR | Delayed |\n|  | AAPL | · | Switzerland | · | CHF | Delayed |\n|  | AAPLE | · | Milan | · | EUR | Real-time |\n|  | AAPL | · | Vienna | · | EUR | Real-time |\n|  | AAPL | · | Warsaw | · | PLN | Delayed |\n|  | AAPLm | · | Buenos Aires | · | ARS | Delayed |\n|  | AAPL | · | Santiago | · | USD | Delayed |\n|  | AAPLCL | · | Santiago | · | CLP | Delayed |\n|  | AAPL34 | · | B3 | · | BRL | Delayed |\n|  | AAPL | · | Toronto | · | CAD | Real-time |\n|  | AAPL | · | Mexico | · | MXN | Delayed |\n|  | AAPLCO | · | Colombia

In [56]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="First find out the stock price of Apple using get stock price tool then use the calculator tool to find out how much will it take to purchase 50 shares?")]})
print(out["messages"][-1].content)

{"query": "current stock price of Apple (AAPL)", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://finance.yahoo.com/quote/AAPL/history", "title": "Apple Inc. (AAPL) Stock Historical Prices & Data", "content": "NasdaqGS - BOATS Real Time Price • USD \n\n# Apple Inc. (AAPL)\n\nTime to act on AAPL?\n\n305.59  -0.34  (-0.11%)\n\nAt close: August 17 at 4:00:01 PM EDT\n\n305.26  -0.33  (-0.11%)\n\nOvernight: 3:35:09 AM EDT\n\nThis price reflects trading activity during the overnight session on the Blue Ocean ATS, available 8 PM to 4 AM ET, Sunday through Thursday, when regular markets are closed.\n\nTime to act on AAPL?\n\nCurrency in USD [...] | Dec 4, 2025 | 284.10 | 284.73 | 278.59 | 280.70 | 279.94 | 43,989,100 |\n| Dec 3, 2025 | 286.20 | 288.62 | 283.30 | 284.15 | 283.38 | 43,538,700 |\n| Dec 2, 2025 | 283.00 | 287.40 | 282.63 | 286.19 | 285.41 | 53,669,500 |\n| Dec 1, 2025 | 278.01 | 283.42 | 276.14 | 283.10 | 282.33 | 46,587,700 |\n| Nov 28, 2025